In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/usdot/flight-delays/airports.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/airlines.csv
/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv


In [20]:
import pandas as pd
import time

# Loading the 2015 Flight Delays dataset
flights = pd.read_csv('/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv', low_memory=False)
df_sample = flights.head(2000).copy()

# Creating the mapping and the Graph (mat)
all_airports = pd.concat([df_sample['ORIGIN_AIRPORT'], df_sample['DESTINATION_AIRPORT']]).unique()
airport_to_idx = {code: i + 1 for i, code in enumerate(all_airports)}
idx_to_airport = {i + 1: code for i, code in enumerate(all_airports)}

mat = []
for index, row in df_sample.iterrows():
    if row['ORIGIN_AIRPORT'] in airport_to_idx and row['DESTINATION_AIRPORT'] in airport_to_idx:
        mat.append([airport_to_idx[row['ORIGIN_AIRPORT']], 
                    airport_to_idx[row['DESTINATION_AIRPORT']], 
                    row['DISTANCE']])

n = len(all_airports)
print(f"Dataset ready: {n} airports mapped.")

Dataset ready: 268 airports mapped.


In [24]:
class MinHeapPQ():
    """Class to execute priority queue functions using a Min-Heap."""
    def __init__(self):
        self.heap_size = 0
        self.heap = []
        
    def get_parent(self, i): return int((i - 1) / 2)
    def get_left(self, i): return int(2 * i + 1)
    def get_right(self, i): return int(2 * i + 2)
    
    def min_heapify(self, i):
        l, r = self.get_left(i), self.get_right(i)
        smallest = l if l < self.heap_size and self.heap[l][1] < self.heap[i][1] else i
        if r < self.heap_size and self.heap[r][1] < self.heap[smallest][1]:
            smallest = r
        if smallest != i:
            self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
            self.min_heapify(smallest)
            
    def insert(self, key_value):
        self.heap.append(key_value)
        self.heap_size = len(self.heap)
        self.decrease_key(self.heap_size - 1)

    def decrease_key(self, index):
        while (index > 0) and (self.heap[self.get_parent(index)][1] > self.heap[index][1]):
            p = self.get_parent(index)
            self.heap[p], self.heap[index] = self.heap[index], self.heap[p]
            index = p
        
    def update_weight(self, key_value):
        for i in range(len(self.heap)):
            if self.heap[i][0] == key_value[0]:
                self.heap[i][1] = key_value[1]
                self.decrease_key(i)
                break

    def extract_min(self):
        if self.heap_size == 0: return None
        root = self.heap[0]
        last = self.heap.pop()
        self.heap_size = len(self.heap)
        if self.heap_size > 0:
            self.heap[0] = last
            self.min_heapify(0)
        return root
            
    def is_empty(self): return self.heap_size == 0

In [32]:
class Dijkstra():
    def __init__(self):
        self.distances, self.is_unique_path, self.parent = {}, {}, {}
        self.pq = MinHeapPQ()
        
    def initialize_single_source(self, n, s):
        for i in range(1, n + 1):
            self.distances[i], self.parent[i], self.is_unique_path[i] = float('inf'), None, 0
        self.distances[s], self.is_unique_path[s] = 0, 1
        
    def contains_vertex(self, v):
        return any(v == item[0] for item in self.pq.heap)
            
    def relax(self, u, v, w):
        if self.distances[v] > (self.distances[u] + w):
            self.distances[v] = self.distances[u] + w
            self.parent[v] = u
            if self.contains_vertex(v): self.pq.update_weight([v, self.distances[v]])
            else: self.pq.insert([v, self.distances[v]])
            self.is_unique_path[v] = self.is_unique_path[u]
        elif self.distances[v] == self.distances[u] + w and self.distances[v] != float('inf'):
            self.is_unique_path[v] = 0
            
    def run_dijkstra(self, n, e, mat, s):
        self.initialize_single_source(n, s)
        discovered = set()
        self.pq.insert([s, 0])
        while not self.pq.is_empty():
            u = self.pq.extract_min()[0]
            discovered.add(u)
            for i in range(e):
                if mat[i][0] == u and (mat[i][1] not in discovered): self.relax(u, mat[i][1], mat[i][2])
                elif mat[i][1] == u and (mat[i][0] not in discovered): self.relax(u, mat[i][0], mat[i][2])
        return [[self.distances[i], self.is_unique_path[i]] for i in range(1, n + 1)]

In [41]:
source_node = 1
start_time = time.time() * 1000 

solver = Dijkstra()
results = solver.run_dijkstra(n, len(mat), mat, source_node)

end_time = time.time() * 1000

print(f"Algorithm: Dijkstra (Greedy Family)")
print(f"Execution Time: {end_time - start_time:.4f} ms")
print("-" * 30)
for i in range(min(10, n)):
    print(f"To {idx_to_airport[i+1]}: {results[i]}")

Algorithm: Dijkstra (Greedy Family)
Execution Time: 43.4495 ms
------------------------------
To ANC: [0, 1]
To LAX: [2376, 0]
To SFO: [2092, 0]
To SEA: [1448, 0]
To LAS: [2305, 0]
To DEN: [2472, 0]
To SLC: [2137, 0]
To PDX: [1542, 0]
To FAI: [261, 1]
To MSP: [2519, 1]


In [34]:
class BellmanFord:
    def __init__(self, n):
        self.n = n
        self.distances = [float('inf')] * (n + 1)

    def run(self, edges, start_node):
        self.distances[start_node] = 0
        
        for i in range(self.n - 1):
            changed = False
            for u, v, w in edges:
                # Check Forward: U -> V
                if self.distances[u] != float('inf') and self.distances[u] + w < self.distances[v]:
                    self.distances[v] = self.distances[u] + w
                    changed = True
                # Check Backward: V -> U (important for undirected flights)
                if self.distances[v] != float('inf') and self.distances[v] + w < self.distances[u]:
                    self.distances[u] = self.distances[v] + w
                    changed = True
            
            # Checking if no distances changed in a full pass
            if not changed:
                print(f"Converged early at iteration {i}")
                break
        
        return self.distances

In [39]:
bf_solver = BellmanFord(n)
source_node = mat[0][0] 

start_bf = time.time() * 1000
bf_results = bf_solver.run(mat, source_node)
end_bf = time.time() * 1000

# Printing the first 5 results to check the distance
for i in range(1, 6):
    print(f"Airport {idx_to_airport[i]}: Distance {bf_results[i]}")

print(f"\nFinal Bellman-Ford Time: {end_bf - start_bf:.2f} ms")

Converged early at iteration 2
Airport ANC: Distance 0
Airport LAX: Distance 2376
Airport SFO: Distance 2092
Airport SEA: Distance 1448
Airport LAS: Distance 2305

Final Bellman-Ford Time: 2.12 ms


In [40]:
import time
source_node = 1 

start_bf = time.time() * 1000 
bf_solver = BellmanFord(n)
bf_results = bf_solver.run(mat, source_node)
end_bf = time.time() * 1000

print(f"Algorithm: Bellman-Ford")
print(f"Execution Time: {end_bf - start_bf:.4f} ms")
print(f"Comparison: Dijkstra (31.45 ms) vs Bellman-Ford ({end_bf - start_bf:.4f} ms)")

Converged early at iteration 2
Algorithm: Bellman-Ford
Execution Time: 2.1978 ms
Comparison: Dijkstra (31.45 ms) vs Bellman-Ford (2.1978 ms)
